# 01 · 车为什么能跟住一条线？

你训练过网络，可能习惯了“输入—预测—loss”。驾驶多了一件事：**本次输出会改变下次输入。**
向左打方向后，位置和朝向改变；下一次控制必须使用新的状态。这就是反馈闭环。

本课目标是独立解释一条实际轨迹：参考从哪里来，动作如何计算，车辆为什么偏离又回来。
假设会 Python、数组和基础三角函数；所需几何在下面解释，先不引入 RL。学习安排见 [单元说明](README.md)。

## 1. 把问题拆成五步

| 步骤 | 本课实际对象 | 应检查的关系 |
|---|---|---|
| 观察 | 自车位置、朝向、速度 | 输入状态与参考路线不同 |
| 选参考 | 固定车道中心线前方的点 | 想去哪里，与现在在哪里不同 |
| 控制 | 误差 → 转向、油门/制动 | 改参考或增益，命令应改变 |
| 执行 | `env.step(applied_action)` | MetaDrive 推进仿真动力学 |
| 再观察 | 执行后的状态 | 反馈到下一步，并写入轨迹 |

本课直接读取模拟器真值状态。MetaDrive 返回的数值观测向量没有输入这个控制器；
相机、检测和状态估计在后续单元接入。当前验证的是几何控制机制。

道路固定为长直道、单车道、无交通。参考车道在 reset 时固定，偏移不会让目标切换到相邻车道。
这是研究控制与延迟的实验条件，尚未包含路口、变道或交互决策。

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from ad_tutorial.driving import DrivingConfig, run_episode, save_episode
OUTPUT = ROOT / "artifacts" / "first_loop"

## 2. 从坐标得到误差

世界坐标 `(x,y)` 描述位置；车道坐标 `(s,e_y)` 描述“沿中心线走了多远”和“偏向哪一侧”。
本课直道沿世界 `+x` 延伸。MetaDrive 横向坐标 **右侧为正**，所以这里 `e_y = y_center - y_ego`。
正转向使车向左转。朝向 theta 用弧度，180°=π rad；车旋转后世界 x 轴不会跟着旋转。

取车道上前方 L 米的点，车辆看向它的角度为：

`bearing = atan2(y_ref - y_ego, x_ref - x_ego)`

`e_heading = wrap(theta_ego - bearing)`

wrap 把角度差映射到 `[-π,π)`，避免跨过 ±π 时跳变。bearing 是**车到参考点的方向**；
车道切线方向在这条直道上恒为 0，两者不总相等。

本课采用容易手算的反馈基线：

`steering = clip(k_y * e_y - k_heading * e_heading, -1, 1)`

`throttle = clip(k_speed * (v_target - v), -1, 1)`

动作无量纲，不是方向盘角度或加速度。k_y 单位 1/m，k_heading 单位 1/rad，k_speed 单位 s/m。
速度用 m/s，6 m/s=21.6 km/h。正油门加速，负值制动。
这是几何反馈启发式，不是标准 Pure Pursuit 或 Stanley 的完整实现。

## 3. 先预测第一步，再运行

车辆在中心线右侧 0.5 m，朝向与直道一致，前视距离 8 m：
bearing=atan2(0.5,8)≈0.06242 rad，heading error≈−0.06242 rad。
在 k_y=0.32、k_heading=0.85 下，转向约 **+0.21306**；静止起步的油门为 **1**。

先写下：偏移换成 −0.5 m，第一步转向是什么符号？k_y 翻倍，误差会在所有时刻减半吗？
区分“第一步命令容易预测”和“整段动力学需要实验”。

In [ ]:
predicted = np.clip(0.32 * 0.5 + 0.85 * np.arctan2(0.5, 8), -1, 1)
print("手算第一步转向:", predicted)
config = DrivingConfig()
result = run_episode(config)
files = save_episode(result, OUTPUT, "lesson1_baseline")
print(result.metrics)
assert np.isclose(result.trace[0]["command_steering"], predicted, atol=1e-5)
display(pd.DataFrame(result.trace).head(6)[[
    "step", "time_s", "before_lateral_error_m", "command_steering",
    "applied_steering", "lateral_error_m", "speed_mps"]])
display(Image(filename=str(files["plot"])))

## 4. 用轨迹检查因果关系

每行保存 before_*（控制输入）、reference_*（目标）、command_*（发出）、applied_*（执行）和
无 before 前缀的状态（执行后）。第 t 行的执行后状态应等于第 t+1 行的输入。
初始状态、固定车道、车身尺寸、有效采样间隔和依赖版本保存在 JSON 配置中。

一步包含 5 次、每次 0.02 秒的物理推进，dt=0.1秒。首行输入在 t=0，执行后状态在 t=0.1秒。
动作经动力学才逐渐改变位置，不会把车辆立即传送到参考点。

In [ ]:
rows = result.trace
r = rows[0]
bearing = np.arctan2(r["reference_y_m"] - r["before_y_m"], r["reference_x_m"] - r["before_x_m"])
heading_error = (r["before_heading_rad"] - bearing + np.pi) % (2*np.pi) - np.pi
recomputed = np.clip(config.lateral_gain * r["before_lateral_error_m"]
                     - config.heading_gain * heading_error, -1, 1)
assert np.isclose(recomputed, r["command_steering"], atol=1e-6)
assert np.allclose([r["x_m"] for r in rows[:-1]], [r["before_x_m"] for r in rows[1:]])
errors = np.array([r["lateral_error_m"] for r in rows])
assert np.isclose(np.mean(np.abs(errors)), result.metrics["mean_abs_lateral_error_m"])
hits = np.flatnonzero(np.abs(errors) < 0.1)
print("首次 |误差| < 0.1m 的采样时间:", rows[hits[0]]["time_s"] if hits.size else "本次未达到")

## 5. 一次只改一个条件

每次先保存一句预测，再运行：

1. **方向**：初始偏移改为 −0.5 m，比较第一步转向和曲线方向。
2. **参考**：保持 +0.5 m，只把前视距离从 8 m 改为 2 m，比较第一步动作、最大误差和终止原因。
3. **迁移**：不再调参，改为 +0.4 m 或 −0.4 m，能否用同一公式解释？

下面是可修改的模板；给每组使用不同文件名以保留证据。

In [ ]:
changed_config = replace(config, initial_lateral_offset_m=-0.5)
changed = run_episode(changed_config)
save_episode(changed, OUTPUT, "lesson1_negative_offset")
display(pd.DataFrame([result.metrics, changed.metrics], index=["+0.5m", "-0.5m"])[[
    "steps", "mean_abs_lateral_error_m", "distance_traveled_m", "outcome"]])

<details><summary>推理与答案：先预测再展开</summary>

负偏移时车在中心线左侧，转向应变负。第一步有对称关系；后续轮胎动力学、边界和数值误差需运行后判断，
不要求轨迹逐位完全镜像。L=2 时 bearing≈0.24498 rad，第一步转向≈0.36823，比 L=8 更大。
更强修正可能更快回正，也可能过冲，不能直接推出“看得近更好”。

增益翻倍直接改变命令的一部分；饱和、heading 项和反馈都会影响后续轨迹。平均误差不会按比例减半。
</details>

## 6. 连接到更大的系统

位置和朝向若来自传感器估计，控制器就要面对误差；参考若由规划器给出，就要考虑道路、障碍和车辆约束。
下一课保留真值输入，只给执行通道增加延迟。

选读 [TUM 第 08 节 Control](https://github.com/TUMFTM/Lecture_ADSE/tree/master/08_control/practice)，限 60–90 分钟：
找出目标速度、当前速度、动作和反馈，逐项对应本课速度环。[后续教材](../../reference/README.md)。